In [ ]:
'''
#Installation

# Clone the ibm/tsfm
! git clone https://github.com/ibm-granite/granite-tsfm.git
! ls

# Change directory. Move inside the tsfm repo.
%cd granite-tsfm
! ls

# Relax requirement for python version < 3.12
! sed -i.orig 's/3\.12/3.13/g' pyproject.toml

# Install the tsfm library
#! pip install ".[notebooks]"
#! python3 -m pip install ".[notebooks]"
! pip3 install ".[notebooks]"
'''

In [ ]:
# Standard
import os, types
import math
import tempfile
import torch
import time

# Third Party
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR
from transformers import EarlyStoppingCallback, Trainer, TrainingArguments, set_seed
from transformers.integrations import INTEGRATION_TO_CALLBACK
import numpy as np
import pandas as pd
from botocore.client import Config
from tsfm_public.toolkit.lr_finder import optimal_lr_finder

from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# tsfm library
from tsfm_public import (
    TimeSeriesPreprocessor,
    TinyTimeMixerForPrediction,
    TrackingCallback,
    count_parameters,
    get_datasets
)
from tsfm_public.toolkit.visualization import plot_predictions
from tsfm_public.toolkit.get_model import get_model

In [ ]:
# Set seed for reproducibility
SEED = 42
set_seed(SEED)

# TTM model branch
TTM_MODEL_PATH = "ibm-granite/granite-timeseries-ttm-r2"

# Forecasting parameters
CONTEXT_LENGTH = 512

# Granite-TTM-R2 supports forecast length upto 720 and Granite-TTM-R1 supports forecast length upto 96
PREDICTION_LENGTH = 96

# Results dir
OUT_DIR = "/TTM/"

In [ ]:
timestamp_column = "DATE"
id_columns = []

from pathlib import Path

dataset_name = "youtube_pedestrian.csv"
data_file = Path.cwd().parents[1] / "data" / "filtered" / dataset_name

df = pd.read_csv(data_file, parse_dates=["DATE"])
print("Loaded:", data_file)


resolution_horizon_map = {
    "100ms": 96,
    "200ms": 48,
    "500ms": 20,
    "1000ms": 10,
    "2000ms": 5,
    "3000ms": 4,    
}


def resample_data_for_ttm(df, resolution):
    df_resampled = df.copy()
    if "DATE" not in df_resampled.columns:
        df_resampled = df_resampled.reset_index()

    df_resampled["DATE"] = pd.to_datetime(df_resampled["DATE"])

    df_resampled = df_resampled.set_index("DATE")

    df_resampled = df_resampled.resample(resolution).mean()
    df_resampled = df_resampled.dropna()
    df_resampled = df_resampled.round().astype(int)

    df_resampled = df_resampled.reset_index()

    return df_resampled

In [ ]:
target_columns = ["mac_dl_brate"]
control_columns = [col for col in df.columns if col not in target_columns]
id_columns = []

column_specifiers = {
    "timestamp_column": timestamp_column,
    "id_columns": [],
    "target_columns": ["mac_dl_brate"],
    "control_columns": ["mac_dl_cqi","mac_dl_mcs","mac_dl_ok","mac_dl_nok",
    ],
}

split_config =  {
    "train": 0.8,
    "test": 0.2,
}

In [ ]:
def zeroshot_eval(
    df_input,
    batch_size,
    context_length=512,
    forecast_length=96,
    cutoff=96,
):
    tsp = TimeSeriesPreprocessor(
        **column_specifiers,
        context_length=context_length,
        prediction_length=forecast_length,
        scaling=True,
        encode_categorical=False,
        scaler_type="minmax",
    )

    zeroshot_model = get_model(
        TTM_MODEL_PATH,
        context_length=context_length,
        prediction_length=forecast_length,
        freq_prefix_tuning=False,
        freq=None,  
        prefer_l1_loss=False,
        prefer_longer_context=True,
    )

    dset_train, dset_valid, dset_test = get_datasets(
        tsp,
        df_input,
        split_config,
        use_frequency_token=zeroshot_model.config.resolution_prefix_tuning,
    )

    raw_timestamps = [item["timestamp"] for item in dset_test]
    test_timestamps = np.array(raw_timestamps)
    test_timestamps = pd.to_datetime(test_timestamps)

    target_column_index = 0

    actuals = np.concatenate(
        [
            item["future_values"].numpy()[
                :, target_column_index : target_column_index + 1
            ]
            for item in dset_test
        ],
        axis=0,
    )

    actuals = actuals.reshape(-1, forecast_length)[:, :cutoff]


    temp_dir = tempfile.mkdtemp()

    zeroshot_trainer = Trainer(
        model=zeroshot_model,
        args=TrainingArguments(
            output_dir=temp_dir,
            per_device_eval_batch_size=batch_size,
            seed=SEED,
            report_to="none",
        ),
    )

    predictions_dict = zeroshot_trainer.predict(dset_test)

    predictions_np = predictions_dict.predictions[0]

    predictions = predictions_np[:, :, target_column_index]
    predictions = predictions.reshape(-1, forecast_length)[:, :cutoff]

    rmse = np.sqrt(mean_squared_error(actuals, predictions))
    mae = mean_absolute_error(actuals, predictions)

    return rmse, mae


In [ ]:
all_ttm_results = []

model_forecast_length = PREDICTION_LENGTH  

for resolution, cutoff in resolution_horizon_map.items():
    print("=" * 70)
    print(f"Running TTM zero-shot for resolution={resolution}, cutoff={cutoff}")

    df_resampled = resample_data_for_ttm(df, resolution)

    rmse_zs, mae_zs = zeroshot_eval(
        df_input=df_resampled,
        context_length=CONTEXT_LENGTH,
        forecast_length=model_forecast_length,  
        cutoff=cutoff,                          
        batch_size=64,                         
    )

    print(f"Resolution {resolution}")
    print(f"Scaled RMSE: {rmse_zs:.3f}")
    print(f"Scaled MAE : {mae_zs:.3f}")

    all_ttm_results.append(
        {
            "Model": "TTM",
            "Variant": "zero_shot",
            "Dataset": "youtube_pedestrian",
            "Resolution": resolution,
            "Prediction Horizon": cutoff,
            "scaled_rmse": rmse_zs,
            "scaled_mae": mae_zs,
        }
    )

ttm_temporal_results = pd.DataFrame(all_ttm_results)

In [ ]:
ttm_temporal_results

In [ ]:
results_dir = Path.cwd().parent / "results" / "metrics" / "temporal_resolution"
results_dir.mkdir(parents=True, exist_ok=True)

metrics_file = results_dir / "ttm_zeroshot.csv"

ttm_temporal_results.to_csv(metrics_file, index=False)

print("Saved metrics to:", metrics_file)

In [ ]:
def fewshot_finetune_eval(
    df_input,
    dataset_name,
    batch_size,
    cutoff,
    learning_rate=None,
    context_length=512,
    forecast_length=PREDICTION_LENGTH,
    fewshot_percent=5,
    freeze_backbone=True,
    num_epochs=30,
    loss="mse",
    quantile=0.5,
    resolution="100ms",
):
    print("-" * 20, f"Running few-shot {fewshot_percent}% for {resolution}", "-" * 20)

    # Data prep
    tsp = TimeSeriesPreprocessor(
        **column_specifiers,
        context_length=context_length,
        prediction_length=forecast_length,
        scaling=True,
        encode_categorical=False,
        scaler_type="minmax",
    )

    # Load model
    if "ett" in str(dataset_name):
        finetune_forecast_model = get_model(
            TTM_MODEL_PATH,
            context_length=context_length,
            prediction_length=forecast_length,
            freq_prefix_tuning=False,
            freq=None,  
            prefer_l1_loss=False,
            prefer_longer_context=True,
            head_dropout=0.7,
            loss=loss,
            quantile=quantile,
        )
    else:
        finetune_forecast_model = get_model(
            TTM_MODEL_PATH,
            context_length=context_length,
            prediction_length=forecast_length,
            freq_prefix_tuning=False,
            freq=None,  
            prefer_l1_loss=False,
            prefer_longer_context=True,
            loss=loss,
            quantile=quantile,
        )

    dset_train, dset_val, dset_test = get_datasets(
        tsp,
        df_input,
        split_config,
        fewshot_fraction=fewshot_percent / 100,
        fewshot_location="first",
        use_frequency_token=finetune_forecast_model.config.resolution_prefix_tuning,
    )

    if freeze_backbone:
        print(
            "Number of params before freezing backbone:",
            count_parameters(finetune_forecast_model),
        )

        for param in finetune_forecast_model.backbone.parameters():
            param.requires_grad = False

        print(
            "Number of params after freezing backbone:",
            count_parameters(finetune_forecast_model),
        )

    # Find optimal learning rate
    if learning_rate is None:
        learning_rate, finetune_forecast_model = optimal_lr_finder(
            finetune_forecast_model,
            dset_train,
            batch_size=batch_size,
        )
        print("OPTIMAL SUGGESTED LEARNING RATE =", learning_rate)

    print(f"Using learning rate = {learning_rate}")

    output_dir = os.path.join(
        OUT_DIR,
        f"finetune_output_{dataset_name}_{resolution}"
    )

    finetune_forecast_args = TrainingArguments(
        output_dir=output_dir,
        overwrite_output_dir=True,
        learning_rate=learning_rate,
        num_train_epochs=num_epochs,
        do_eval=True,
        eval_strategy="epoch",
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        dataloader_num_workers=8,
        report_to="none",
        save_strategy="epoch",
        logging_strategy="epoch",
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        seed=SEED,
    )

    early_stopping_callback = EarlyStoppingCallback(
        early_stopping_patience=10,
        early_stopping_threshold=1e-5,
    )

    tracking_callback = TrackingCallback()

    optimizer = AdamW(finetune_forecast_model.parameters(), lr=learning_rate)

    scheduler = OneCycleLR(
        optimizer,
        learning_rate,
        epochs=num_epochs,
        steps_per_epoch=math.ceil(len(dset_train) / batch_size),
    )

    finetune_forecast_trainer = Trainer(
        model=finetune_forecast_model,
        args=finetune_forecast_args,
        train_dataset=dset_train,
        eval_dataset=dset_val,
        callbacks=[early_stopping_callback, tracking_callback],
        optimizers=(optimizer, scheduler),
    )

    finetune_forecast_trainer.remove_callback(
        INTEGRATION_TO_CALLBACK["codecarbon"]
    )

    # Fine-tune
    finetune_forecast_trainer.train()

    # Evaluation
    print("+" * 20, f"Test MSE after few-shot {fewshot_percent}% fine-tuning", "+" * 20)

    finetune_forecast_trainer.model.loss = "mse"

    fewshot_output = finetune_forecast_trainer.evaluate(dset_test)
    print(fewshot_output)
    print("+" * 60)

    raw_timestamps = [item["timestamp"] for item in dset_test]
    test_timestamps = np.array(raw_timestamps)
    test_timestamps = pd.to_datetime(test_timestamps)

    target_column_index = 0

    actuals = np.concatenate(
        [
            item["future_values"].numpy()[
                :, target_column_index : target_column_index + 1
            ]
            for item in dset_test
        ],
        axis=0,
    )

    actuals = actuals.reshape(-1, forecast_length)[:, :cutoff]

    predictions_dict = finetune_forecast_trainer.predict(dset_test)

    predictions_np = predictions_dict.predictions[0]

    backbone_embedding = predictions_dict.predictions[1]

    predictions = predictions_np[:, :, target_column_index]
    predictions = predictions.reshape(-1, forecast_length)[:, :cutoff]

    rmse = np.sqrt(mean_squared_error(actuals, predictions))
    mae = mean_absolute_error(actuals, predictions)

    return rmse, mae

In [ ]:
all_ttm_finetune_results = []

model_forecast_length = PREDICTION_LENGTH

for resolution, cutoff in resolution_horizon_map.items():
    print("=" * 70)
    print(f"Running TTM fine-tuning for resolution={resolution}, cutoff={cutoff}")

    df_resampled = resample_data_for_ttm(df, resolution)

    df_resampled = df_resampled.astype(
        {col: "float32" for col in df_resampled.columns if col != "DATE"}
    )

    rmse_ft, mae_ft = fewshot_finetune_eval(
        df_input=df_resampled,
        dataset_name="youtube_pedestrian",
        context_length=CONTEXT_LENGTH,
        forecast_length=model_forecast_length,
        cutoff=cutoff,
        batch_size=128,
        fewshot_percent=5,
        learning_rate=None,
        resolution=resolution,
    )

    print(f"Resolution {resolution}")
    print(f"Fine-tuning RMSE: {rmse_ft:.3f}")
    print(f"Fine-tuning MAE : {mae_ft:.3f}")

    all_ttm_finetune_results.append(
        {
            "Model": "TTM",
            "Variant": "fine_tuning",
            "Dataset": "youtube_pedestrian",
            "Resolution": resolution,
            "Prediction Horizon": model_forecast_length,
            "scaled_rmse": rmse_ft,
            "scaled_mae": mae_ft,
        }
    )

ttm_finetune_temporal_results = pd.DataFrame(all_ttm_finetune_results)

In [ ]:
ttm_finetune_temporal_results

In [ ]:
results_dir = Path.cwd().parent / "results" / "metrics" / "temporal_resolution"
results_dir.mkdir(parents=True, exist_ok=True)

metrics_file = results_dir / "ttm_finetune.csv"

ttm_finetune_temporal_results.to_csv(metrics_file, index=False)

print("Saved metrics to:", metrics_file)